In [60]:
import pandas as pd, geopandas as gpd, folium
from folium.plugins import FeatureGroupSubGroup, GroupedLayerControl, MarkerCluster
import json
from datetime import datetime
import glob
import os
import html 
from collections import defaultdict


In [61]:
# ===== 0. pilot input =====
try:
    pilot_lat_str = input("Pilot latitude (decimal degrees) [44.0]: ").strip()
    pilot_lon_str = input("Pilot longitude (decimal degrees) [-79.0]: ").strip()
    pilot_lat = float(pilot_lat_str) if pilot_lat_str else 44.0
    pilot_lon = float(pilot_lon_str) if pilot_lon_str else -79.0
except ValueError:
    raise SystemExit("numeric lat/lon required")
print(f"Pilot location set to: {pilot_lat:.6f}, {pilot_lon:.6f}")

nm_to_m   = 1852 #nautical mile to meters
zoom_init = 8  # initial zoom level for the map


Pilot latitude (decimal degrees) [44.0]:  
Pilot longitude (decimal degrees) [-79.0]:  


Pilot location set to: 44.000000, -79.000000


In [62]:
# ===== 1. load airport csv =====
df = pd.read_csv("https://davidmegginson.github.io/ourairports-data/airports.csv")
df = df[df["iso_country"] == "CA"].copy()  # Only Canadian aerodromes
df.dropna(subset=["ident", "latitude_deg", "longitude_deg"], inplace=True)
gdf = gpd.GeoDataFrame(df,
        geometry=gpd.points_from_xy(df.longitude_deg, df.latitude_deg),
        crs="EPSG:4326")

In [63]:
# gdf # dataframe of airports.csv


In [64]:
# ===== 2. icon lookup =====
ICON_STYLE = {
    "heliport":       {"icon":"helicopter","color":"green"},
    "seaplane_base":  {"icon":"ship",      "color":"cadetblue"},
    "small_airport":  {"icon":"circle",    "color":"gray"},
    "medium_airport": {"icon":"plane",     "color":"blue"},
    "large_airport":  {"icon":"plane",     "color":"darkblue"},
    "default":        {"icon":"map-marker","color":"lightgray"},
}

# ===== 3. base map =====
# m = folium.Map(location=[pilot_lat, pilot_lon], zoom_start=zoom_init, tiles="OpenStreetMap")
m = folium.Map(location=[pilot_lat, pilot_lon], zoom_start=zoom_init, tiles="Cartodb Positron") 

In [44]:
# m

In [67]:
zoom_start = 10
m = folium.Map(location=[pilot_lat, pilot_lon], tiles="OpenStreetMap", zoom_start=zoom_start)

In [68]:
m

In [49]:
# ===== 4. province/type layers =====
province_parents, layers_dict = {}, {}
layer_dict = defaultdict(list)
province_names = sorted(gdf["iso_region"].unique())
airport_types = sorted(gdf["type"].unique())

for prov in province_names:
    fg = folium.FeatureGroup(name=prov, show=False).add_to(m)
    province_parents[prov] = fg
    layers_dict[prov] = []

for prov, prov_df in gdf.groupby("iso_region"):
    for atype, sub in prov_df.groupby("type"):
        child = FeatureGroupSubGroup(province_parents[prov], atype, show=False, overlay=True)  # show=False hides by default
        cluster = MarkerCluster().add_to(child)
        style = ICON_STYLE.get(atype, ICON_STYLE["default"])
        for _, row in sub.iterrows():
            lat, lon = row.geometry.y, row.geometry.x
            folium.Marker(
                [lat, lon],
                tooltip=f"{row['ident']} • {row['name']}",
                popup=f"{row['ident']} – {row['name']}<br>Lat: {lat:.6f}<br>Lon: {lon:.6f}",
                icon=folium.Icon(color=style["color"], icon=style["icon"], prefix="fa")
            ).add_to(cluster)
        child.add_to(m)
        layers_dict[prov].append(child)
# 
for prov in province_names:
    for atype in airport_types:
        sub = gdf[(gdf["iso_region"] == prov) & (gdf["type"] == atype)]
        if sub.empty:
            continue
        layer_name = f"{prov} • {atype}"
        fg = folium.FeatureGroup(name=layer_name)
        for _, row in sub.iterrows():
            lat, lon = row.geometry.y, row.geometry.x
            style = ICON_STYLE.get(atype, ICON_STYLE["default"])
            folium.Marker(
                [lat, lon],
                tooltip=f"{row['ident']} • {row['name']}",
                popup=f"{row['ident']} – {row['name']}<br>Lat: {lat:.6f}<br>Lon: {lon:.6f}",
                icon=folium.Icon(color=style["color"], icon=style["icon"], prefix="fa")
            ).add_to(fg)
        fg.add_to(m)
        layer_dict[prov].append(fg)
# 
# When adding GroupedLayerControl, set exclusive_groups to the list of provinces:
GroupedLayerControl(
    groups=layers_dict,
    exclusive_groups=[],  # Only one province visible at a time
    collapsed=True
).add_to(m)


In [51]:
# m

In [52]:
# m

In [69]:
# ===== 4. province/type layers =====
province_parents, layers_dict = {}, {}
layer_dict = defaultdict(list)
province_names = sorted(gdf["iso_region"].unique())
airport_types = sorted(gdf["type"].unique())

for prov in province_names:
    fg = folium.FeatureGroup(name=prov).add_to(m)
    province_parents[prov] = fg
    layers_dict[prov] = []

for prov, prov_df in gdf.groupby("iso_region"):
    for atype, sub in prov_df.groupby("type"):
        child = FeatureGroupSubGroup(province_parents[prov], atype, show=False)
        cluster = MarkerCluster().add_to(child)
        style = ICON_STYLE.get(atype, ICON_STYLE["default"])
        
        for _, row in sub.iterrows():
            lat, lon = row.geometry.y, row.geometry.x
            folium.Marker(
                [lat, lon],
                tooltip=f"{row['ident']} • {row['name']}",
                popup=f"{row['ident']} – {row['name']}<br>Lat: {lat:.6f}<br>Lon: {lon:.6f}",
                icon=folium.Icon(color=style["color"], icon=style["icon"], prefix="fa")
            ).add_to(cluster)
        
        child.add_to(m)
        layers_dict[prov].append(child)

# Correctly mapping provinces to their layers for GroupedLayerControl
grouped_layers = {prov: layers_dict[prov] for prov in province_names}

# Adding GroupedLayerControl
GroupedLayerControl(
    groups=grouped_layers,
    exclusive_groups=[],  # Change if you want to allow multiple selections
    collapsed=False
).add_to(m)


In [59]:
m

KeyboardInterrupt: 